In [1]:
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import jieba
from torch.utils.data import Dataset, DataLoader
import time
import random
import tqdm

In [2]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
def NormalizeEnglishString(s):
    s=s.lower().strip()
    s=re.sub(r'([.,!?;:\'"\-()%$])',r' \1',s)
    s=re.sub(r'[^a-zA-Z.,!?;:\'"\-()%$]+',r' ',s)
    return s

In [4]:
with open('../data/zh-en.txt','r',encoding='utf-8') as f:
    lines=f.readlines()

In [5]:
my_pairs=[]
for line in lines:
    zh,en=line.strip().split('\t')
    en=NormalizeEnglishString(en)
    my_pairs.append((zh,en))

In [6]:
print('zh-en pairs:',len(my_pairs))

zh-en pairs: 493085


In [7]:
SOS_token=0
EOS_token=1
chinese_word2index={'SOS':SOS_token,'EOS':EOS_token}
english_word2index={'SOS':SOS_token,'EOS':EOS_token}

In [8]:
for zh,_ in my_pairs:
    for word in jieba.lcut(zh):
        if word not in chinese_word2index:
            chinese_word2index[word]=len(chinese_word2index)

for _,en in my_pairs:
    for word in en.split():
        if word not in english_word2index:
            english_word2index[word]=len(english_word2index)

Building prefix dict from the default dictionary ...
Dumping model to file cache C:\Users\User\AppData\Local\Temp\jieba.cache
Loading model cost 1.533 seconds.
Prefix dict has been built successfully.


In [11]:
chinese_word_count=len(chinese_word2index)
english_word_count=len(english_word2index)
print('chinese_word_count:', chinese_word_count)
print('english_word_count:', english_word_count)

chinese_word_count: 68838
english_word_count: 39105


In [15]:
torch.save({'chinese_word2index':chinese_word2index,
            'english_word2index':english_word2index,
            'chinese_word_count':chinese_word_count,
            'english_word_count':english_word_count},
           '../models/word2index_and_word_count.pt')

In [16]:
word2index_and_word_count = torch.load('../models/word2index_and_word_count.pt')
chinese_word2index = word2index_and_word_count['chinese_word2index']
english_word2index = word2index_and_word_count['english_word2index']
chinese_word_count = word2index_and_word_count['chinese_word_count']
english_word_count = word2index_and_word_count['english_word_count']

In [18]:
class My_Dataset(Dataset):
    def __init__(self, pairs, chinese_word2index, english_word2index):
        self.pairs = pairs
        self.chinese_word2index = chinese_word2index
        self.english_word2index = english_word2index

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        zh, en = self.pairs[idx]
        x = [self.chinese_word2index[word] for word in jieba.lcut(zh)] + [EOS_token]
        y = [self.english_word2index[word] for word in en.split()] + [EOS_token]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

In [27]:
class C2E_Translator_Encoder(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1):
        super(C2E_Translator_Encoder,self).__init__()
        self.input_size=input_size
        self.hidden_size=hidden_size
        self.num_layers=num_layers
        self.embedding=nn.Embedding(input_size,hidden_size)
        self.gru=nn.GRU(hidden_size,hidden_size,num_layers,batch_first=True)

    def forward(self,x,hidden):
        embedded=self.embedding(x)
        output,hidden=self.gru(embedded,hidden)
        return output,hidden

    def init_hidden(self,batch_size):
        return torch.zeros(self.num_layers,batch_size,self.hidden_size,device=device)

In [24]:
class C2E_Translator_Decoder_Attention(nn.Module):
    def __init__(self,hidden_size,output_size,num_layers=1):
        super(C2E_Translator_Decoder_Attention,self).__init__()
        self.hidden_size=hidden_size
        self.output_size=output_size
        self.num_layers=num_layers
        self.embedding=nn.Embedding(output_size,hidden_size)
        self.query_combine=nn.Linear(hidden_size*2,hidden_size)
        self.attn_combine=nn.Linear(hidden_size*2,hidden_size)
        self.gru=nn.GRU(hidden_size,hidden_size,num_layers,batch_first=True)
        self.out=nn.Linear(hidden_size,output_size)
        self.dropout=nn.Dropout(0.1)

    def forward(self,input,hidden,encoder_outputs):
        embedded=self.embedding(input)
        embedded=self.dropout(embedded)
        query=self.query_combine(torch.cat((embedded,hidden[-1].unsqueeze(0)),dim=-1))
        permuted_encoder_outputs=encoder_outputs.permute(0,2,1)
        attn_weights=torch.bmm(query,permuted_encoder_outputs)
        attn_weights=F.softmax(attn_weights,dim=-1)
        attn_applied=torch.bmm(attn_weights,encoder_outputs)
        attn_combined=self.attn_combine(torch.cat((query,attn_applied),dim=-1))
        attn_combined=F.relu(attn_combined)
        output,hidden=self.gru(attn_combined,hidden)
        output=self.out(output)
        return output,hidden,attn_weights

    def init_hidden(self,batch_size):
        return torch.zeros(self.num_layers,batch_size,self.hidden_size,device=device)

In [28]:
encoder=C2E_Translator_Encoder(input_size=chinese_word_count,hidden_size=256,num_layers=1).to(device)
decoder=C2E_Translator_Decoder_Attention(hidden_size=256,output_size=english_word_count,num_layers=1).to(device)

In [31]:
batch_size=1
my_dataloader=DataLoader(My_Dataset(my_pairs,chinese_word2index,english_word2index),batch_size=batch_size,shuffle=True)
criterion=nn.CrossEntropyLoss()
optimizer_encoder=torch.optim.Adam(encoder.parameters(),lr=0.0001)
optimizer_decoder=torch.optim.Adam(decoder.parameters(),lr=0.0001)

In [32]:
epochs=10
interval=1000
teacher_forcing_ratio=0.5

In [36]:
for epoch in range(epochs):
    encoder.train()
    decoder.train()
    start_time=time.time()
    iter_num=0
    total_loss=0.0
    interval_loss=0.0
    for x,y in tqdm.tqdm(my_dataloader):
        iter_num+=1
        x=x.to(device)
        y=y.to(device)
        batch_size=x.size(0)
        optimizer_encoder.zero_grad()
        optimizer_decoder.zero_grad()
        encoder_hidden=encoder.init_hidden(batch_size)
        encoder_outputs,encoder_hidden=encoder(x,encoder_hidden)
        decoder_input=torch.tensor([[SOS_token]*batch_size],dtype=torch.long,device=device).transpose(0,1)
        decoder_hidden=encoder_hidden
        loss=0.0
        use_teacher_forcing=True if random.random()<teacher_forcing_ratio else False
        if use_teacher_forcing:
            for di in range(y.size(1)):
                decoder_output,decoder_hidden,decoder_attention=decoder(decoder_input,decoder_hidden,encoder_outputs)
                loss+=criterion(decoder_output.squeeze(1),y[:,di])
                decoder_input=y[:,di].unsqueeze(1)
        else:
            for di in range(y.size(1)):
                decoder_output,decoder_hidden,decoder_attention=decoder(decoder_input,decoder_hidden,encoder_outputs)
                topv,topi=decoder_output.topk(1)
                decoder_input=topi.squeeze(-1).detach()
                loss+=criterion(decoder_output.squeeze(1),y[:,di])
                if (topi==EOS_token).all():
                    break
        loss.backward()
        optimizer_encoder.step()
        optimizer_decoder.step()
        total_loss+=loss.item()/y.size(1)
        interval_loss+=loss.item()/y.size(1)

torch.save(encoder.state_dict(), '../models/encoder.pt')
torch.save(decoder.state_dict(), '../models/decoder.pt')

  0%|          | 46/493085 [00:21<64:01:27,  2.14it/s]


KeyboardInterrupt: 